# Data Quality Assessment

## Import Libraries

In [ ]:
%matplotlib inline
%pip install fg-data-profiling missingno --quiet

import sys, os

sys.path.insert(0, os.path.abspath(""))

from utils.config import *
from utils.loader import load_tables
from utils.dqa import run_dqa
from utils.domain_rules import get_domain_rules
from utils.schema_guard import SchemaContract
import missingno as msno
from data_profiling import ProfileReport

os.makedirs(DQA_REPORT_DIR, exist_ok=True)

print("Libraries imported and settings configured")

## Loading Reconciled Database

In [ ]:
dfs = load_tables("reconciled", normalize_cols="lower")

# Register schemas for post-operation validation
contract = SchemaContract()
for name, df in dfs.items():
    contract.register(name, df)

print(f"\nRegistered {len(contract.registered_tables)} table schemas.")

## Definition of Domain Rules

In [ ]:
domain_rules = get_domain_rules()

print("Domain Rules configured (with referential integrity via closures).")
print(f"Tables covered: {list(domain_rules.keys())}")

## DQA, Profiling and Plotting

In [ ]:
for table_name, df in dfs.items():
    print(f"\n{'='*50}\nANALYSIS: {table_name}\n{'='*50}")

    # 1. Run full DQA using the shared run_dqa() function
    dqa = run_dqa(df, table_name, domain_rules, dfs_ref=dfs)

    scorecard = dqa.scorecard()
    print(f"Overall DQ Score: {dqa.overall_score():.4%}\n")
    print(scorecard[["Dimension", "Score", "Issues", "Status"]].to_string(index=False))

    # Save Scorecard
    scorecard.to_csv(DQA_REPORT_DIR / f"Scorecard_{table_name}.csv", index=False)

    # 2. YData Profiling
    print(f"\nGenerating HTML Profile for {table_name}...")
    profile = ProfileReport(df, title=f"Data Quality Profile - {table_name}", explorative=True, minimal=True)
    profile.to_file(DQA_REPORT_DIR / f"Profile_{table_name}.html")

    # 3. Plotting (Missingno + Scorecard)
    fig, axes = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={"width_ratios": [1.5, 1]})

    # 3a. Missing Values Matrix
    msno.matrix(df, ax=axes[0], color=(0.4, 0.4, 0.4), sparkline=False, fontsize=10)
    axes[0].set_title(f"Missing Values Matrix: {table_name.upper()}", fontsize=14, fontweight="bold", pad=20, loc="left", color="#333333")

    # 3b. Scorecard Bar Chart
    sc = scorecard.sort_values(by="Score", ascending=True)
    colors = ["#2ecc71" if s >= 0.95 else ("#f1c40f" if s >= 0.80 else "#e74c3c") for s in sc["Score"]]
    bars = axes[1].barh(sc["Dimension"], sc["Score"], color=colors, height=0.6, alpha=0.85, zorder=3)
    axes[1].set_xlim(0, 1.15)

    axes[1].axvline(0.95, color="#cccccc", linestyle="--", linewidth=1.2, zorder=0)
    axes[1].text(0.90, -0.75, "Target (95%)", color="#888888", fontsize=10, ha="left", va="top")

    axes[1].axvline(0.80, color="#cccccc", linestyle="--", linewidth=1.2, zorder=0)
    axes[1].text(0.85, -0.75, "Warn (80%)", color="#888888", fontsize=10, ha="right", va="top")

    for bar in bars:
        width = bar.get_width()
        axes[1].text(width + 0.015, bar.get_y() + bar.get_height() / 2, f"{width:.2%}", va="center", fontsize=11, fontweight="semibold", color="#333333")

    axes[1].set_title(f"Data Quality Scores: {table_name.upper()}", fontsize=14, fontweight="bold", pad=20, loc="left", color="#333333")
    axes[1].set_xticks([])
    for spine in ["top", "right", "bottom", "left"]:
        axes[1].spines[spine].set_visible(False)
    axes[1].tick_params(axis="y", length=0, labelsize=11, labelcolor="#555555")

    plt.subplots_adjust(wspace=0.3)
    plt.savefig(DQA_REPORT_DIR / f"Plot_{table_name}.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## Post-DQA Schema Validation

In [ ]:
# Verify that the DQA process did not inadvertently modify any table
report = contract.validate_all(dfs)
if report["failures"] == 0:
    print("Schema validation PASSED - no structural changes after DQA.")
else:
    print("Schema validation FAILED - unexpected structural changes detected!")
    for entry in report["details"]:
        if entry["status"] != "PASS":
            print(f"  {entry}")